# 🚀 TinyTransformer (~238M) Kaggle TPU v3-8 (8-Core Distributed Pretraining)

**High-Throughput Pretraining Engine for Kaggle TPU VM v3-8 (8 Cores x 16 GB HBM = 128 GB Total HBM)**

### 📐 Model Architecture & Specifications
- **Parameters:** ~237.85 Million (BF16 native)
- **Backbone:** 24 Transformer layers, hidden dim 640, 10 attention heads, SwiGLU 3.0x multiplier
- **Engram Memory:** 80M global (post-L16) + two 10M tables at L5 & L11 (99.5M params)
- **Critic BS-Detector:** 4-layer auxiliary discriminator (9.2M params) for adversarial alignment
- **Optimizers:** Dual Keller Jordan Muon (2D internal weights) + AdamW (Embeds, Norms, Engram tables)
- **Batch Geometry (All 8 Cores):** Micro-batch 4 x Accum 1 x 8 Cores = **32,768 tokens/step**
- **Target Acceleration:** Kaggle TPU VM v3-8 (8 Cores in parallel via `torch_xla` multiprocessing)
- **Projected Throughput:** **~120,000 – 160,000 tokens/sec**
- **Repository:** `https://github.com/logicstillloading/TinyTransformer`

---
## ⚙️ Step 1: Verify Kaggle TPU VM v3-8 (8 Cores Active)
Verifies that Kaggle notebook accelerator is set to **TPU VM v3-8** and tests PyTorch/XLA core availability.

In [ ]:
import os
import sys
import torch

print(f"[*] PyTorch Version: {torch.__version__}")

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    print(f"[+] torch_xla version: {torch_xla.__version__}")
    print(f"[+] Device: {xm.xla_device()}")
    print(f"[+] Hardware Type: {xm.xla_device_hw(xm.xla_device())}")
    print(f"[+] TPU Cores Available: {xm.xrt_world_size()}")
except Exception as e:
    print(f"[!] TPU check warning: {e}")
    print("[*] Ensure Accelerator is set to 'TPU VM v3-8' in Kaggle Notebook Settings (right sidebar).")

---
## 📥 Step 2: Clone TinyTransformer Repository & Install Dependencies
Clones the codebase directly into Kaggle working environment and installs runtime requirements.

In [ ]:
import os

if not os.path.exists("/kaggle/working/TinyTransformer"):
    !git clone https://github.com/logicstillloading/TinyTransformer.git /kaggle/working/TinyTransformer

%cd /kaggle/working/TinyTransformer
!git pull origin main

# Install dependencies for multi-core tokenization and dataset streaming
!pip install -q einops sentencepiece datasets gdown psutil

---
## 📂 Step 3: Setup Google Drive Checkpoint (Resume Seamlessly)
You can resume your exact progress (Step 8,750+ / ~286M tokens seen) using your Google Drive checkpoint.

### Method A: Direct Google Drive Link (Automatic Download via `gdown`)
Paste your Google Drive share link (set to *"Anyone with the link can view"*) or file ID below:

In [ ]:
import os
import shutil

# Paste your Google Drive shared file link or ID (e.g., https://drive.google.com/file/d/...)
GDRIVE_CHECKPOINT_URL = ""  # <-- PASTE YOUR GOOGLE DRIVE LINK HERE IF AVAILABLE

os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
DEST_PATH = "/kaggle/working/checkpoints/checkpoint_latest.pt"

if GDRIVE_CHECKPOINT_URL.strip():
    import gdown
    print(f"[*] Downloading checkpoint from Google Drive...")
    gdown.download(url=GDRIVE_CHECKPOINT_URL, output=DEST_PATH, fuzzy=True)
    if os.path.exists(DEST_PATH):
        print(f"[+] Checkpoint downloaded successfully! ({os.path.getsize(DEST_PATH)/(1024**3):.2f} GB)")
else:
    # Check if checkpoint was attached via Kaggle Input Dataset
    found = False
    for root, dirs, files in os.walk("/kaggle/input"):
        if "checkpoint_latest.pt" in files:
            src = os.path.join(root, "checkpoint_latest.pt")
            print(f"[+] Found checkpoint in Kaggle Dataset: {src}")
            shutil.copy2(src, DEST_PATH)
            found = True
            break
    if not found:
        print("[*] No checkpoint provided yet. If you want to start fresh from Step 0, you can proceed directly to training.")
        print("[*] Or paste your Google Drive link in GDRIVE_CHECKPOINT_URL above and re-run this cell.")

---
## ⚡ Step 4: Pre-Tokenize Full 750M Tokens into Kaggle NVMe Disk (Zero-RAM Mode)
Pre-tokenizes all 6 streams directly onto Kaggle scratch storage so all 8 TPU cores read with zero network latency and zero host RAM overhead.

In [ ]:
%cd /kaggle/working/TinyTransformer

# Runs high-speed parallel NVMe pre-tokenization across all 6 streams (~750M tokens)
!python -u data/download_and_tokenize.py --shard-dir /kaggle/working/TinyTransformer/data/shards --tokens 750000000

---
## 🔥 Step 5: Launch 8-Core Distributed Pretraining on Kaggle TPU v3-8
Runs all **8 TPU v3 cores** in parallel via PyTorch/XLA. Each core processes micro-batch 4 with **zero accumulation delay**, achieving massive **~130,000 tokens/sec** throughput.

In [ ]:
%cd /kaggle/working/TinyTransformer

# Launch 8-core distributed pretraining with atomic checkpoints and 5x mini-inference interval (1250 steps)
!python -u train.py \
    --tpu-cores 8 \
    --batch-size 4 \
    --grad-accum 1 \
    --dtype bfloat16 \
    --checkpoint-dir /kaggle/working/checkpoints \
    --log-dir /kaggle/working/logs \
    --save-interval 250 \
    --infer-interval 1250 \
    --resume

---
## 📊 Step 6: Live Loss & Perplexity Telemetry Audit
Audits real-time loss progression, perplexity decay, and learning rate curves.

In [ ]:
%cd /kaggle/working/TinyTransformer
!python plot_metrics.py --log-dir /kaggle/working/logs